# Task 4 — Who Speaks What (language identification)

Predict the language of short support messages: `ru`, `kaz` or `eng`.

Statement: [`../qualification/task4_Who_Speaks_What.md`](../qualification/task4_Who_Speaks_What.md)
Reasoning behind every choice here: [`task4_who_speaks_what.md`](./task4_who_speaks_what.md)

**Before running:** put `train.csv` and `test.csv` next to this notebook, or edit `TRAIN` / `TEST` below.
Dataset links are in the statement (Google Drive).

In [ ]:
TRAIN = "train.csv"      # columns: text, label
TEST  = "test.csv"       # column:  text
OUT   = "solution.csv"   # columns: id, label

USE_AUGMENTATION = True  # manufacture code-switched rows; see the .md for why
RANDOM_STATE = 42

In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report

train = pd.read_csv(TRAIN)
test = pd.read_csv(TEST)
print("train", train.shape, "| test", test.shape)
print(train.label.value_counts())
train.head()

## Augmentation — the part that actually wins

Kazakh has nine Cyrillic letters Russian does not use. A model can reach high accuracy by
looking only for those letters, and then collapses on messages that happen not to contain one.

We manufacture exactly those hard cases: some Kazakh rows get their special letters replaced by
the nearest Russian letter, and some rows get transliterated into Latin script. The label never
changes, so this is label-preserving augmentation.

In [ ]:
KAZ_TO_RUS = {
    "Ә": "А", "ә": "а", "І": "И", "і": "и", "Ң": "Н", "ң": "н",
    "Ғ": "Г", "ғ": "г", "Ү": "У", "ү": "у", "Ұ": "У", "ұ": "у",
    "Ө": "О", "ө": "о", "Қ": "К", "қ": "к", "Һ": "Х", "һ": "х",
}

CYR_TO_LAT = {
    "а":"a","б":"b","в":"v","г":"g","д":"d","е":"e","ё":"e","ж":"zh","з":"z","и":"i",
    "й":"y","к":"k","л":"l","м":"m","н":"n","о":"o","п":"p","р":"r","с":"s","т":"t",
    "у":"u","ф":"f","х":"h","ц":"ts","ч":"ch","ш":"sh","щ":"sch","ъ":"","ы":"y","ь":"",
    "э":"e","ю":"yu","я":"ya",
}

def kazakh_to_russian(text: str) -> str:
    return "".join(KAZ_TO_RUS.get(ch, ch) for ch in str(text))

def translit(text: str) -> str:
    out = []
    for ch in str(text):
        low = ch.lower()
        rep = CYR_TO_LAT.get(low)
        if rep is None:
            out.append(ch)
        else:
            out.append(rep.upper() if ch.isupper() else rep)
    return "".join(out)

def augment(df: pd.DataFrame, seed: int = RANDOM_STATE) -> pd.DataFrame:
    """Half the kaz rows lose their special letters; half of those are then Latinised.
    Half the ru rows are Latinised. Labels are untouched."""
    kaz = df[df.label == "kaz"]
    ru  = df[df.label == "ru"]
    eng = df[df.label == "eng"]

    kaz_a = kaz.sample(frac=0.5, random_state=seed).copy()
    kaz_b = kaz.drop(kaz_a.index)
    kaz_a["text"] = kaz_a.text.map(kazakh_to_russian)
    kaz_a1 = kaz_a.sample(frac=0.5, random_state=seed).copy()
    kaz_a2 = kaz_a.drop(kaz_a1.index)
    kaz_a1["text"] = kaz_a1.text.map(translit)

    ru_a = ru.sample(frac=0.5, random_state=seed).copy()
    ru_b = ru.drop(ru_a.index)
    ru_a["text"] = ru_a.text.map(translit)

    return pd.concat([kaz_a1, kaz_a2, kaz_b, ru_a, ru_b, eng], ignore_index=True)

train_aug = augment(train) if USE_AUGMENTATION else train.copy()
print("rows:", len(train), "->", len(train_aug))
print(train_aug.label.value_counts())

## Model

Character n-grams, not words. Language identification on short text is decided by letter
patterns and endings, and `char_wb` keeps n-grams inside word boundaries so it never invents
features that span a space.

In [ ]:
def build_model(analyzer="char_wb", ngram=(1, 4), C=10.0):
    return make_pipeline(
        TfidfVectorizer(analyzer=analyzer, ngram_range=ngram,
                        sublinear_tf=True, min_df=2, max_features=200_000),
        LogisticRegression(C=C, solver="saga", max_iter=1000,
                           random_state=RANDOM_STATE, n_jobs=-1),
    )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# The statement names no metric. The official reference notebook grades with
# classification_report and tunes on f1_macro, so macro-F1 is what we optimise.
for name, model in [
    ("char_wb 1-4", build_model("char_wb", (1, 4))),
    ("char    2-5", build_model("char",    (2, 5))),
    ("word   1-1",  build_model("word",    (1, 1))),
]:
    s = cross_val_score(model, train_aug.text, train_aug.label,
                        cv=cv, scoring="f1_macro", n_jobs=-1)
    print(f"{name}: macro-F1 {s.mean():.4f} +/- {s.std(ddof=1)/np.sqrt(len(s)):.4f}")

In [ ]:
model = build_model("char_wb", (1, 4))
model.fit(train_aug.text, train_aug.label)

# Sanity check on the ORIGINAL, un-augmented training data.
print(classification_report(train.label, model.predict(train.text), zero_division=0))

## Write the submission\n\nExact filename, exact columns, ids `0..n-1` in `test.csv` order, no index column.

In [ ]:
pred = model.predict(test.text)

sub = pd.DataFrame({"id": range(len(pred)), "label": pred})
sub.to_csv(OUT, index=False)

assert list(sub.columns) == ["id", "label"]
assert len(sub) == len(test), f"{len(sub)} rows, expected {len(test)}"
assert set(sub.label) <= {"ru", "kaz", "eng"}, set(sub.label)
assert sub.id.tolist() == list(range(len(sub)))
print(f"{OUT}: {len(sub)} rows OK")
print(sub.label.value_counts())
sub.head()